# FERRUS OSSEUS — Fregate 04
## FLOTTE FERRUS | AD MAJOREM GLORIAM IMPERATORIS

**Pipeline batch** : Tous les avatars T-pose de `IN/` → FBX rigés dans `OUT/`

```
Drive/FLOTTE-FERRUS/04_FERRUS_OSSEUS/
  IN/
    avatar_P1.glb      ┐
    avatar_P2.fbx      │  →  osseus_core.py (Blender headless)
    avatar_P3.obj      ┘
  OUT/
    avatar_rigged_avatar_P1_r15.fbx
    avatar_rigged_avatar_P2_r15.fbx
    avatar_rigged_avatar_P3_r15.fbx
    rapport_osseus_batch.json
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 1 — CONFIGURATION
# Seule cellule a editer. Tout le reste est automatique.
# ══════════════════════════════════════════════════════════════════════════

# ── Template squelette ───────────────────────────────────────────────────
TEMPLATE = "r15"                 # r15 | mixamo | deepmotion
#   r15        : 15 bones Roblox R15       → pour FERRUS ANIMUS
#   mixamo     : 26 bones Mixamo           → pour Adobe Mixamo / Unity
#   deepmotion : 52 bones DeepMotion       → pour FERRUS ANIMUS (full body)

# ── Acces GitHub ─────────────────────────────────────────────────────────
GITHUB_TOKEN = "TON_TOKEN_ICI"   # Remplacer par ton token GitHub
GITHUB_REPO  = "kioka8877-ux/FLOTTE-FERRUS"

# ── Blender sur Drive (optionnel) ────────────────────────────────────────
# Si tu as deja Blender installe sur ton Drive (depuis un autre projet),
# indique le chemin exact vers l'executable blender.
# Laisser vide ("") pour recherche automatique sur Drive.
# Mettre None pour forcer l'installation apt (ignorer Drive).
BLENDER_DRIVE_PATH = ""          # ex: "/content/drive/MyDrive/blender-3.6/blender"
BLENDER_MIN_VERSION = (3, 0)     # Version minimale requise

# ══════════════════════════════════════════════════════════════════════════
# NE PAS MODIFIER EN DESSOUS
# ══════════════════════════════════════════════════════════════════════════
DRIVE_REPO  = "/content/drive/MyDrive/FLOTTE-FERRUS"
FREGATE_DIR = f"{DRIVE_REPO}/04_FERRUS_OSSEUS"
IN_DIR      = f"{FREGATE_DIR}/IN"
OUT_DIR     = f"{FREGATE_DIR}/OUT"
SCRIPT_PATH = f"{FREGATE_DIR}/codebase/osseus_core.py"
SUPPORTED   = {".glb", ".gltf", ".obj", ".fbx"}

print("[OSSEUS] Configuration OK")
print(f"  Template : {TEMPLATE}")
print(f"  Drive IN : {IN_DIR}")
print(f"  Drive OUT: {OUT_DIR}")
print()
print("[OSSEUS] Executer C2 → C3 → C4 → C5 dans l'ordre.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 2 — SETUP DRIVE + REPO
# Monte Drive, clone le repo dessus (ou git pull si deja present).
# Apres cette cellule : Drive/FLOTTE-FERRUS/ est pret avec tout le code.
# ══════════════════════════════════════════════════════════════════════════

from google.colab import drive
import subprocess, os

def run(cmd, cwd=None, silent=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if not silent:
        out = (r.stdout + r.stderr).strip()
        if out:
            for line in out.split("\n")[-8:]:
                if line.strip():
                    print(f"  {line}")
    return r

# ── 1. Monter Drive ───────────────────────────────────────────────────────
print("[OSSEUS] Montage Google Drive...")
drive.mount("/content/drive")
print("[OSSEUS] Drive monte.")

# ── 2. Clone ou git pull sur Drive ────────────────────────────────────────
git_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"

if not os.path.exists(f"{DRIVE_REPO}/.git"):
    print(f"\n[OSSEUS] Premier lancement — clone du repo sur Drive...")
    run(f"git clone {git_url} '{DRIVE_REPO}'")
    print("[OSSEUS] Clone termine.")
else:
    print(f"\n[OSSEUS] Repo present — git pull (derniere version du code)...")
    run(f"git remote set-url origin {git_url}", cwd=DRIVE_REPO, silent=True)
    run("git pull --rebase", cwd=DRIVE_REPO)
    print("[OSSEUS] Code a jour.")

# ── 3. Creer IN/ et OUT/ si absents ──────────────────────────────────────
for d in [IN_DIR, OUT_DIR, f"{FREGATE_DIR}/codebase/docs"]:
    os.makedirs(d, exist_ok=True)

# ── 4. Verifier le script ─────────────────────────────────────────────────
if not os.path.exists(SCRIPT_PATH):
    raise RuntimeError(f"Script manquant : {SCRIPT_PATH}")

# ── 5. Bilan ─────────────────────────────────────────────────────────────
in_files = [f for f in os.listdir(IN_DIR)
            if os.path.splitext(f)[1].lower() in SUPPORTED]
print(f"\n[OSSEUS] Drive pret.")
print(f"  Repo   : {DRIVE_REPO}")
print(f"  IN/    : {len(in_files)} avatar(s) detecte(s)")
for f in sorted(in_files):
    size = os.path.getsize(os.path.join(IN_DIR, f)) / (1024*1024)
    print(f"    {f} ({size:.2f} Mo)")
if not in_files:
    print("    (vide — deposer des avatars dans Drive/IN/ avant de continuer)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 3 — BLENDER : RECHERCHE DRIVE → INSTALLATION SI ABSENT
# Cherche un Blender existant sur Drive avant de telecharger.
# Si trouve et version OK → utilise directement depuis Drive.
# Si absent → installe via apt (une fois par session).
# ══════════════════════════════════════════════════════════════════════════

import subprocess, os, re

BLENDER_BIN = None  # Sera resolu ci-dessous

def get_blender_version(path):
    """Retourne (major, minor) ou None si invalide.
    Tente d'abord d'executer le binaire.
    Si ca echoue (Drive FUSE, permissions), parse la version depuis le chemin."""
    r = subprocess.run(f"'{path}' --version", shell=True,
                       capture_output=True, text=True)
    if r.returncode == 0:
        m = re.search(r"Blender\s+(\d+)\.(\d+)", r.stdout + r.stderr)
        if m:
            return (int(m.group(1)), int(m.group(2)))
    # Fallback : extraire la version depuis le nom du dossier
    # ex : blender-4.2, blender-3.6.5, blender_4.1
    m = re.search(r"blender[-_](\d+)\.(\d+)", path, re.IGNORECASE)
    if m:
        return (int(m.group(1)), int(m.group(2)))
    return None

def blender_ok(path):
    """True si l'executable existe et la version est suffisante."""
    if not path or not os.path.isfile(path):
        return False
    ver = get_blender_version(path)
    if ver and ver >= BLENDER_MIN_VERSION:
        return True
    if ver:
        print(f"  Version trop ancienne : {ver[0]}.{ver[1]} < {BLENDER_MIN_VERSION[0]}.{BLENDER_MIN_VERSION[1]}")
    return False

# ── 1. Chemin explicite dans la config ───────────────────────────────────
if BLENDER_DRIVE_PATH and BLENDER_DRIVE_PATH != "":
    print(f"[OSSEUS] Test Blender (chemin config) : {BLENDER_DRIVE_PATH}")
    if blender_ok(BLENDER_DRIVE_PATH):
        BLENDER_BIN = BLENDER_DRIVE_PATH
        ver = get_blender_version(BLENDER_BIN)
        print(f"[OSSEUS] Blender {ver[0]}.{ver[1]} trouve sur Drive (chemin config).")
    else:
        print(f"[OSSEUS] Chemin config invalide, recherche automatique...")

# ── 2. Recherche automatique sur Drive ───────────────────────────────────
if not BLENDER_BIN and BLENDER_DRIVE_PATH != None:
    print("[OSSEUS] Recherche Blender sur Drive...")
    search_cmd = (
        "find /content/drive/MyDrive -maxdepth 4 "
        "-name 'blender' -type f 2>/dev/null"
    )
    r = subprocess.run(search_cmd, shell=True, capture_output=True, text=True)
    candidates = [p.strip() for p in r.stdout.strip().split("\n") if p.strip()]

    if candidates:
        print(f"  Candidats trouves : {len(candidates)}")
        for candidate in candidates:
            ver = get_blender_version(candidate)
            if ver:
                print(f"  Test : {candidate}  →  version {ver[0]}.{ver[1]}")
            else:
                print(f"  Test : {candidate}  →  version non detectee")
            if blender_ok(candidate):
                BLENDER_BIN = candidate
                print(f"[OSSEUS] Blender {ver[0]}.{ver[1]} recupere depuis Drive : {BLENDER_BIN}")
                break
        if not BLENDER_BIN:
            print("  Aucun candidat valide.")
    else:
        print("  Aucun binaire 'blender' trouve sur Drive.")

# ── 3. Verifier si deja installe dans la session ─────────────────────────
if not BLENDER_BIN:
    r_sys = subprocess.run("which blender", shell=True, capture_output=True, text=True)
    if r_sys.returncode == 0:
        sys_path = r_sys.stdout.strip()
        if blender_ok(sys_path):
            BLENDER_BIN = sys_path
            ver = get_blender_version(BLENDER_BIN)
            print(f"[OSSEUS] Blender {ver[0]}.{ver[1]} deja installe (session) : {BLENDER_BIN}")

# ── 4. Installation apt si rien trouve ────────────────────────────────────
if not BLENDER_BIN:
    print("[OSSEUS] Installation Blender via apt (~2-3 min)...")
    print("[OSSEUS] Mise a jour des listes de paquets...")
    r_update = subprocess.run(
        "apt-get update -qq 2>&1 | tail -3",
        shell=True, capture_output=True, text=True
    )
    if r_update.stdout.strip():
        print(f"  {r_update.stdout.strip()[-200:]}")

    print("[OSSEUS] Installation...")
    r_apt = subprocess.run(
        "apt-get install -y --fix-missing blender 2>&1 | tail -5",
        shell=True, capture_output=True, text=True
    )
    print(r_apt.stdout[-400:] if r_apt.stdout else r_apt.stderr[-400:])

    r_which = subprocess.run("which blender", shell=True, capture_output=True, text=True)
    if r_which.returncode == 0 and blender_ok(r_which.stdout.strip()):
        BLENDER_BIN = r_which.stdout.strip()
        ver = get_blender_version(BLENDER_BIN)
        print(f"[OSSEUS] Blender {ver[0]}.{ver[1]} installe : {BLENDER_BIN}")
    else:
        raise RuntimeError(
            "Blender non disponible.\n"
            "  Option 1 : specifier BLENDER_DRIVE_PATH dans Cell 1\n"
            "  Option 2 : relancer la cellule (les miroirs apt fluctuent)\n"
            "  Option 3 : dans une cellule separee, executer : !apt-get update && apt-get install -y blender"
        )

print(f"\n[OSSEUS] Blender pret : {BLENDER_BIN}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 4 — SCAN IN/ + CONFIRMATION AVANT LANCEMENT
# Liste tous les avatars dans IN/ et confirme avant de lancer le batch.
# ══════════════════════════════════════════════════════════════════════════

import os

input_files = sorted([
    f for f in os.listdir(IN_DIR)
    if os.path.splitext(f)[1].lower() in SUPPORTED
])

if not input_files:
    print(f"[OSSEUS] IN/ est vide.")
    print(f"  Deposer les avatars dans : {IN_DIR}")
    print(f"  Formats supportes : .glb  .gltf  .obj  .fbx")
    raise RuntimeError("Aucun avatar a traiter dans IN/")

print(f"[OSSEUS] {len(input_files)} avatar(s) a riger :")
print(f"  Template : {TEMPLATE}")
print()
for i, f in enumerate(input_files, 1):
    path = os.path.join(IN_DIR, f)
    size = os.path.getsize(path) / (1024*1024)
    stem = os.path.splitext(f)[0]
    out  = f"avatar_rigged_{stem}_{TEMPLATE}.fbx"
    print(f"  [{i}] {f} ({size:.2f} Mo)  →  OUT/{out}")

print()
print("[OSSEUS] Executer C5 pour lancer le batch.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 5 — PIPELINE BATCH
# Traite tous les avatars de IN/ l'un apres l'autre.
# Chaque sortie FBX est sauvegardee directement sur Drive/OUT/.
# ══════════════════════════════════════════════════════════════════════════

import os, subprocess, json
from pathlib import Path

batch_rapport = []
success_count = 0
error_count   = 0

print(f"[OSSEUS] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"[OSSEUS] BATCH — {len(input_files)} avatar(s) | template={TEMPLATE}")
print(f"[OSSEUS] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

for idx, filename in enumerate(input_files, 1):
    stem        = Path(filename).stem
    input_path  = os.path.join(IN_DIR, filename)
    output_file = f"avatar_rigged_{stem}_{TEMPLATE}.fbx"
    output_path = os.path.join(OUT_DIR, output_file)
    report_path = os.path.join(OUT_DIR, f"rapport_osseus_{stem}.json")

    print(f"\n[OSSEUS] [{idx}/{len(input_files)}] {filename}")

    cmd = (
        f"'{BLENDER_BIN}' --background --python '{SCRIPT_PATH}' -- "
        f"--input   '{input_path}' "
        f"--output  '{output_path}' "
        f"--template {TEMPLATE} "
        f"--report  '{report_path}'"
    )

    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    # Logs OSSEUS seulement
    all_output  = result.stdout + result.stderr
    osseus_logs = [l for l in all_output.split("\n") if l.startswith("[OSSEUS]")]
    for line in osseus_logs:
        print(f"  {line}")

    # Lire le rapport JSON
    if os.path.exists(report_path):
        with open(report_path) as f:
            rpt = json.load(f)
    else:
        rpt = {"status": "ERROR", "error": "Rapport JSON non genere",
               "input": filename, "template": TEMPLATE}

    if rpt.get('status') == 'SUCCESS':
        success_count += 1
        size = os.path.getsize(output_path) / (1024*1024) if os.path.exists(output_path) else 0
        print(f"  OK : {output_file} ({size:.2f} Mo) | {rpt.get('bones_count','-')} bones | weights={'AUTO' if rpt.get('auto_weights') else 'ENVELOPE'}")
    else:
        error_count += 1
        print(f"  ECHEC : {rpt.get('error', 'erreur inconnue')}")
        if result.returncode != 0:
            print(f"  Blender logs : {all_output[-800:]}")

    batch_rapport.append(rpt)

# ── Rapport global ───────────────────────────────────────────────────────
global_report = {
    "total":    len(input_files),
    "success":  success_count,
    "errors":   error_count,
    "template": TEMPLATE,
    "avatars":  batch_rapport,
}
global_report_path = os.path.join(OUT_DIR, "rapport_osseus_batch.json")
with open(global_report_path, "w") as f:
    json.dump(global_report, f, indent=2, ensure_ascii=False)

print(f"\n[OSSEUS] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"[OSSEUS] BATCH TERMINE")
print(f"[OSSEUS]   Total   : {len(input_files)}")
print(f"[OSSEUS]   Succes  : {success_count}")
print(f"[OSSEUS]   Echecs  : {error_count}")
print(f"[OSSEUS]   Rapport : Drive/OUT/rapport_osseus_batch.json")
print(f"[OSSEUS] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 6 — RAPPORT BATCH DETAILLE
# ══════════════════════════════════════════════════════════════════════════

import json, os

with open(global_report_path) as f:
    gr = json.load(f)

print(f"[OSSEUS] RAPPORT BATCH — {gr['success']}/{gr['total']} OK")
print()
print(f"  {'Avatar':<35} {'Status':<10} {'Bones':<8} {'Mo':<8} {'Weights'}")
print(f"  {'-'*35} {'-'*10} {'-'*8} {'-'*8} {'-'*10}")

for av in gr['avatars']:
    name    = os.path.basename(av.get('input', '-'))[:34]
    status  = av.get('status', '?')
    bones   = str(av.get('bones_count', '-'))
    size    = str(av.get('output_size_mb', '-'))
    weights = 'AUTO' if av.get('auto_weights') else ('ENVELOPE' if av.get('status') == 'SUCCESS' else '-')
    err     = f" [{av.get('error','')}]" if status == 'ERROR' else ''
    print(f"  {name:<35} {status:<10} {bones:<8} {size:<8} {weights}{err}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELLULE 7 — TRANSFERT VERS FERRUS ANIMUS (optionnel)
# Copie tous les FBX rigues dans Drive/01_FERRUS_ANIMUS/IN/
# ══════════════════════════════════════════════════════════════════════════

import os, shutil

TRANSFERT_VERS_ANIMUS = False   # Passer a True pour copie auto
ANIMUS_IN_DIR = f"{DRIVE_REPO}/01_FERRUS_ANIMUS/IN"

if TRANSFERT_VERS_ANIMUS:
    os.makedirs(ANIMUS_IN_DIR, exist_ok=True)
    transferts = []
    for av in gr['avatars']:
        if av.get('status') == 'SUCCESS':
            stem = Path(av['input']).stem
            fbx  = f"avatar_rigged_{stem}_{TEMPLATE}.fbx"
            src  = os.path.join(OUT_DIR, fbx)
            dst  = os.path.join(ANIMUS_IN_DIR, fbx)
            if os.path.exists(src):
                shutil.copy2(src, dst)
                transferts.append(fbx)
                print(f"[OSSEUS] Transfere : {fbx}")
    print(f"\n[OSSEUS] {len(transferts)} FBX transferes vers FERRUS ANIMUS IN/")
    print(f"[OSSEUS] Ouvrir 01_FERRUS_ANIMUS/main_ferrus.ipynb pour continuer.")
else:
    print("[OSSEUS] Transfert desactive (TRANSFERT_VERS_ANIMUS=False).")
    print(f"[OSSEUS] Les FBX rigues sont dans : {OUT_DIR}")